# Chapter 10: The Five Platonic Solids

**Source span.** Introduction to Geometry, printed pages 148-159, PDF pages 166-177. The PDF pages were inspected as page images because the scan has no extractable text. This notebook uses the source only for orientation and terminology; all prose, diagrams, code, and artifacts here are original.

**Chapter goal.** Build the five Platonic solids from inspectable data: vertices, edges, faces, Schlaefli symbols, radii, angles, planar drawings, and reciprocal pairs. The target is not only to see the solids, but to identify which claims are metric, which are combinatorial, and which are topological.

The source chapter moves from construction families to drawings, then to Euler's formula, radii and angles, and reciprocal polyhedra. The notebook follows the same mathematical spine. Every major object is represented twice: once as a visual model that a reader can inspect, and once as a count, formula, or residual that can fail if the construction is wrong.


In [ ]:
from __future__ import annotations

from collections import OrderedDict, defaultdict
from itertools import product
from pathlib import Path
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial import ConvexHull
import sympy as sp
import trimesh

from IPython.display import Markdown, display

CHAPTER_NO = 10
HERE = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [HERE, *HERE.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not locate the Introduction-to-Geometry book root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, display_artifact, write_csv, write_json

ART_ROOT = BOOK_ROOT / "artifacts" / "chapter-10"
ART_DIRS = {kind: ART_ROOT / kind for kind in ["figures", "html", "checks", "tables", "data"]}
for directory in ART_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

STALE_ARTIFACTS = [
    ART_DIRS["figures"] / "concept_configuration.svg",
    ART_DIRS["figures"] / "parameter_experiment.svg",
]
for stale_path in STALE_ARTIFACTS:
    if stale_path.exists():
        stale_path.unlink()

SOURCE_SPAN = {
    "printed_pages": "148-159",
    "pdf_pages": "166-177",
    "source_map_note": "AGENTS.md gives pdf_page = printed_page + 18 for body pages.",
}

storyboard = {
    "chapter_goal": "Make the five Platonic solids inspectable through constructions, planar drawings, Euler counts, radii/angle formulas, and reciprocal polyhedra.",
    "source_span_read": SOURCE_SPAN,
    "concept_inventory": [
        "right regular pyramids, prisms, and antiprisms",
        "perspective drawings, nets, and Schlegel diagrams as different records of one incidence structure",
        "Euler formula V - E + F = 2 and the two edge-counting relations qV = 2E = pF",
        "classification by the solid-angle inequality (p-2)(q-2) < 4",
        "circumradius, midradius, inradius, central angle, and dihedral angle of {p,q}",
        "reciprocal polyhedra with {p,q} paired to {q,p}",
    ],
    "library_routing": [
        {"concept": "3D solids and reciprocal overlays", "library": "Plotly", "why": "interactive rotation exposes face adjacency and dual placement"},
        {"concept": "mesh counts and edge/face incidence", "library": "Trimesh", "why": "mesh topology checks guard the hand-built coordinates"},
        {"concept": "planar maps and Schlegel graphs", "library": "NetworkX", "why": "Euler's formula is a graph/map invariant"},
        {"concept": "classification and radii formulas", "library": "SymPy + NumPy", "why": "exact formulas plus numeric residuals catch algebra mistakes"},
        {"concept": "static proof diagrams and tables", "library": "Matplotlib", "why": "durable labeled PNGs are best for planar proof scaffolds"},
    ],
}
write_json(ART_DIRS["checks"] / "visual-storyboard.json", storyboard)

display(Markdown(
    f"Loaded chapter {CHAPTER_NO} artifacts at `{ART_ROOT.relative_to(BOOK_ROOT).as_posix()}`. "
    f"Removed stale scaffold artifacts: {sum(not p.exists() for p in STALE_ARTIFACTS)}/{len(STALE_ARTIFACTS)}."
))


## Computational Translation Guide

A regular polyhedron is compact enough to draw but strict enough to audit. In this notebook:

- a **construction** is a coordinate recipe for vertices and cyclic face lists;
- a **drawing** is either a 3D model, a planar net-like record, or a Schlegel graph;
- a **topological claim** is an Euler characteristic check;
- a **regularity claim** is an equality of lengths, angles, face valences, or radii;
- a **reciprocity claim** is an exchange of vertices and faces, tested by face-center duals.

The chapter's notation `{p,q}` means that each face is a regular `p`-gon and `q` faces meet at each vertex. The code keeps `p` and `q` visible rather than hiding them in canned mesh assets, because the formulas in this chapter are really statements about the pair `{p,q}`.


In [ ]:
def regular_ngon(n: int, radius: float = 1.0, z: float = 0.0, rotation: float = 0.0) -> np.ndarray:
    angles = rotation + np.linspace(0, 2 * np.pi, n, endpoint=False)
    return np.column_stack([radius * np.cos(angles), radius * np.sin(angles), np.full(n, z)])


def edge_pairs(faces: list[list[int]]) -> list[tuple[int, int]]:
    edges: set[tuple[int, int]] = set()
    for face in faces:
        for a, b in zip(face, face[1:] + face[:1]):
            edges.add(tuple(sorted((int(a), int(b)))))
    return sorted(edges)


def triangulate_faces(faces: list[list[int]]) -> np.ndarray:
    triangles: list[list[int]] = []
    for face in faces:
        if len(face) < 3:
            continue
        for k in range(1, len(face) - 1):
            triangles.append([face[0], face[k], face[k + 1]])
    return np.array(triangles, dtype=int)


def mesh_stats(vertices: np.ndarray, faces: list[list[int]]) -> dict[str, int | bool]:
    triangles = triangulate_faces(faces)
    mesh = trimesh.Trimesh(vertices=vertices, faces=triangles, process=False)
    return {
        "V": int(len(vertices)),
        "E": int(len(edge_pairs(faces))),
        "F": int(len(faces)),
        "chi": int(len(vertices) - len(edge_pairs(faces)) + len(faces)),
        "triangulated_euler": int(mesh.euler_number),
        "watertight_triangulation": bool(mesh.is_watertight),
    }


def edge_lengths(vertices: np.ndarray, faces: list[list[int]]) -> np.ndarray:
    return np.array([np.linalg.norm(vertices[a] - vertices[b]) for a, b in edge_pairs(faces)])


def scale_to_edge_one(vertices: np.ndarray, faces: list[list[int]]) -> np.ndarray:
    lengths = edge_lengths(vertices, faces)
    return vertices / float(np.median(lengths))


def face_centers(vertices: np.ndarray, faces: list[list[int]]) -> np.ndarray:
    return np.array([vertices[face].mean(axis=0) for face in faces])


def plotly_edges_trace(vertices: np.ndarray, faces: list[list[int]], name: str, color: str = "#1f2937", width: float = 4.0) -> go.Scatter3d:
    xs: list[float | None] = []
    ys: list[float | None] = []
    zs: list[float | None] = []
    for a, b in edge_pairs(faces):
        xs.extend([float(vertices[a, 0]), float(vertices[b, 0]), None])
        ys.extend([float(vertices[a, 1]), float(vertices[b, 1]), None])
        zs.extend([float(vertices[a, 2]), float(vertices[b, 2]), None])
    return go.Scatter3d(x=xs, y=ys, z=zs, mode="lines", line=dict(color=color, width=width), name=name, showlegend=False)


def plotly_mesh_trace(vertices: np.ndarray, faces: list[list[int]], name: str, color: str, opacity: float = 0.48) -> go.Mesh3d:
    triangles = triangulate_faces(faces)
    return go.Mesh3d(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=triangles[:, 0], j=triangles[:, 1], k=triangles[:, 2],
        color=color, opacity=opacity, flatshading=True, name=name, showlegend=False,
    )


def add_polyhedron(fig: go.Figure, vertices: np.ndarray, faces: list[list[int]], row: int, col: int, title: str, color: str) -> None:
    fig.add_trace(plotly_mesh_trace(vertices, faces, title, color), row=row, col=col)
    fig.add_trace(plotly_edges_trace(vertices, faces, title + " edges"), row=row, col=col)


def equal_axis_layout(fig: go.Figure) -> go.Figure:
    scene_update = dict(aspectmode="data", xaxis_visible=False, yaxis_visible=False, zaxis_visible=False)
    for scene_name in ["scene", "scene2", "scene3", "scene4", "scene5", "scene6"]:
        if scene_name in fig.layout:
            fig.update_layout(**{scene_name: scene_update})
    return fig


## Pyramids, Prisms, And Antiprisms

The chapter begins by constructing familiar families before isolating the five regular cases. A right regular `n`-gonal prism has two regular `n`-gons joined by rectangles; if the height is chosen correctly, those rectangles are squares. A right regular `n`-gonal pyramid has one regular `n`-gon and `n` isosceles triangles; for `n < 6` the height can be chosen so the lateral triangles are equilateral. An `n`-gonal antiprism rotates the top polygon halfway between the lower vertices and joins the two levels by `2n` triangles.

The interactive artifact below uses edge length one wherever the regular side faces demand it. The inspection target is the local pattern at a vertex: prism vertices see `n`-gon plus two rectangles, pyramid base vertices see an `n`-gon plus two triangles, and antiprism vertices see one `n`-gon plus three triangles. The check is deliberately combinatorial: all three families should have Euler characteristic two.


In [ ]:
def pyramid(n: int, equilateral_sides: bool = True) -> tuple[np.ndarray, list[list[int]]]:
    base_radius = 0.5 / math.sin(math.pi / n)
    base = regular_ngon(n, base_radius, z=0.0, rotation=math.pi / n)
    if equilateral_sides:
        height_sq = 1.0 - base_radius**2
        if height_sq <= 0:
            raise ValueError("equilateral lateral faces only close for n < 6")
        height = math.sqrt(height_sq)
    else:
        height = 1.0
    vertices = np.vstack([base, [0.0, 0.0, height]])
    apex = n
    faces = [list(reversed(range(n)))] + [[i, (i + 1) % n, apex] for i in range(n)]
    return vertices, faces


def prism(n: int, square_sides: bool = True) -> tuple[np.ndarray, list[list[int]]]:
    base_radius = 0.5 / math.sin(math.pi / n)
    height = 1.0 if square_sides else 0.7
    bottom = regular_ngon(n, base_radius, z=-height / 2, rotation=math.pi / n)
    top = regular_ngon(n, base_radius, z=height / 2, rotation=math.pi / n)
    vertices = np.vstack([bottom, top])
    faces = [list(reversed(range(n))), list(range(n, 2 * n))]
    faces += [[i, (i + 1) % n, n + (i + 1) % n, n + i] for i in range(n)]
    return vertices, faces


def antiprism(n: int, equilateral_triangles: bool = True) -> tuple[np.ndarray, list[list[int]]]:
    base_radius = 0.5 / math.sin(math.pi / n)
    horizontal_lateral = 2 * base_radius * math.sin(math.pi / (2 * n))
    height = math.sqrt(max(1.0 - horizontal_lateral**2, 0.0)) if equilateral_triangles else 0.65
    bottom = regular_ngon(n, base_radius, z=-height / 2, rotation=0.0)
    top = regular_ngon(n, base_radius, z=height / 2, rotation=math.pi / n)
    vertices = np.vstack([bottom, top])
    faces = [list(reversed(range(n))), list(range(n, 2 * n))]
    for i in range(n):
        faces.append([i, (i + 1) % n, n + i])
        faces.append([n + (i - 1) % n, i, n + i])
    return vertices, faces


family_specs = OrderedDict([
    ("pentagonal pyramid", pyramid(5)),
    ("pentagonal prism", prism(5)),
    ("pentagonal antiprism", antiprism(5)),
])

family_rows = []
for family_name, (vertices, faces) in family_specs.items():
    stats = mesh_stats(vertices, faces)
    family_rows.append({"family": family_name, **stats})
    assert stats["chi"] == 2

construction_html = ART_DIRS["html"] / "construction-families-pyramid-prism-antiprism.html"
fig = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "scene"}, {"type": "scene"}, {"type": "scene"}]],
    subplot_titles=["pyramid", "prism", "antiprism"],
)
for col, (family_name, (vertices, faces)) in enumerate(family_specs.items(), start=1):
    add_polyhedron(fig, vertices, faces, 1, col, family_name, ["#ef4444", "#0ea5e9", "#10b981"][col - 1])
fig.update_layout(title="Right regular construction families with edge-length checks", height=520, margin=dict(l=0, r=0, t=60, b=0))
equal_axis_layout(fig)
fig.write_html(construction_html, include_plotlyjs=True, full_html=True)

construction_counts_csv = ART_DIRS["tables"] / "construction-family-counts.csv"
construction_checks_json = ART_DIRS["checks"] / "construction-family-checks.json"
write_csv(construction_counts_csv, family_rows)
write_json(construction_checks_json, {"chapter": CHAPTER_NO, "families": family_rows, "all_euler_two": all(row["chi"] == 2 for row in family_rows)})

display_artifact(construction_html, width=920)
display(Markdown("Construction family counts: " + ", ".join(f"{row['family']} has V={row['V']}, E={row['E']}, F={row['F']}" for row in family_rows)))


## Drawings And Models

A solid can be drawn in several ways, and each drawing keeps different information. A perspective view preserves a sense of shape but hides some incidences. A net records faces and fold edges but sacrifices the final three-dimensional angles. A Schlegel diagram projects the graph of vertices and edges into the plane and counts the outside as the initial face. That last move is why Euler's formula can be proved by studying planar maps.

The first artifact in this section is a rotatable model gallery for the five Platonic solids. The second is a small cube pipeline: perspective model, net, and Schlegel map. The third is a Schlegel-style graph gallery for all five solids. It is intentionally graph-first: the learner should inspect which vertices, edges, and face regions survive the projection rather than treating the drawing as a metric picture.


In [ ]:
def order_face_vertices(vertices: np.ndarray, ids: list[int], normal: np.ndarray) -> list[int]:
    pts = vertices[ids]
    center = pts.mean(axis=0)
    u = pts[0] - center
    u = u / np.linalg.norm(u)
    v = np.cross(normal, u)
    v = v / np.linalg.norm(v)
    angles = np.arctan2((pts - center) @ v, (pts - center) @ u)
    return [ids[i] for i in np.argsort(angles)]


def grouped_hull_faces(vertices: np.ndarray) -> list[list[int]]:
    hull = ConvexHull(vertices)
    groups: dict[tuple[float, ...], set[int]] = defaultdict(set)
    normals: dict[tuple[float, ...], np.ndarray] = {}
    for simplex, equation in zip(hull.simplices, hull.equations):
        key = tuple(np.round(equation, 6))
        groups[key].update(int(i) for i in simplex)
        normals[key] = equation[:3]
    return [order_face_vertices(vertices, sorted(ids), normals[key]) for key, ids in groups.items()]


def hull_triangle_faces(vertices: np.ndarray) -> list[list[int]]:
    return [list(map(int, simplex)) for simplex in ConvexHull(vertices).simplices]


def build_platonic_meshes() -> OrderedDict[str, dict]:
    phi = (1 + math.sqrt(5)) / 2
    meshes: OrderedDict[str, dict] = OrderedDict()

    tetra_vertices = np.array([[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]], dtype=float)
    tetra_faces = [[0, 1, 2], [0, 3, 1], [0, 2, 3], [1, 3, 2]]
    meshes["tetrahedron"] = {"symbol": "{3,3}", "p": 3, "q": 3, "vertices": scale_to_edge_one(tetra_vertices, tetra_faces), "faces": tetra_faces}

    cube_vertices = np.array([
        [-1, -1, -1], [1, -1, -1], [1, 1, -1], [-1, 1, -1],
        [-1, -1, 1], [1, -1, 1], [1, 1, 1], [-1, 1, 1],
    ], dtype=float)
    cube_faces = [[0, 1, 2, 3], [4, 7, 6, 5], [0, 4, 5, 1], [1, 5, 6, 2], [2, 6, 7, 3], [3, 7, 4, 0]]
    meshes["cube"] = {"symbol": "{4,3}", "p": 4, "q": 3, "vertices": scale_to_edge_one(cube_vertices, cube_faces), "faces": cube_faces}

    octa_vertices = np.array([[1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0], [0, 0, 1], [0, 0, -1]], dtype=float)
    octa_faces = [[4, 0, 2], [4, 2, 1], [4, 1, 3], [4, 3, 0], [5, 2, 0], [5, 1, 2], [5, 3, 1], [5, 0, 3]]
    meshes["octahedron"] = {"symbol": "{3,4}", "p": 3, "q": 4, "vertices": scale_to_edge_one(octa_vertices, octa_faces), "faces": octa_faces}

    icosa_vertices = []
    for y in [-1, 1]:
        for z in [-phi, phi]:
            icosa_vertices.append([0, y, z])
    for x in [-1, 1]:
        for y in [-phi, phi]:
            icosa_vertices.append([x, y, 0])
    for x in [-phi, phi]:
        for z in [-1, 1]:
            icosa_vertices.append([x, 0, z])
    icosa_vertices = np.array(icosa_vertices, dtype=float)
    icosa_faces = hull_triangle_faces(icosa_vertices)
    meshes["icosahedron"] = {"symbol": "{3,5}", "p": 3, "q": 5, "vertices": scale_to_edge_one(icosa_vertices, icosa_faces), "faces": icosa_faces}

    dodeca_vertices = []
    for signs in product([-1, 1], repeat=3):
        dodeca_vertices.append(signs)
    for y in [-1 / phi, 1 / phi]:
        for z in [-phi, phi]:
            dodeca_vertices.append([0, y, z])
    for x in [-1 / phi, 1 / phi]:
        for y in [-phi, phi]:
            dodeca_vertices.append([x, y, 0])
    for x in [-phi, phi]:
        for z in [-1 / phi, 1 / phi]:
            dodeca_vertices.append([x, 0, z])
    dodeca_vertices = np.array(dodeca_vertices, dtype=float)
    dodeca_faces = grouped_hull_faces(dodeca_vertices)
    meshes["dodecahedron"] = {"symbol": "{5,3}", "p": 5, "q": 3, "vertices": scale_to_edge_one(dodeca_vertices, dodeca_faces), "faces": dodeca_faces}

    return meshes


PLATONIC = build_platonic_meshes()

platonic_rows = []
for name, data in PLATONIC.items():
    vertices, faces, p, q = data["vertices"], data["faces"], data["p"], data["q"]
    stats = mesh_stats(vertices, faces)
    lengths = edge_lengths(vertices, faces)
    row = {
        "solid": name,
        "symbol": data["symbol"],
        "p": p,
        "q": q,
        **stats,
        "edge_length_min": float(lengths.min()),
        "edge_length_max": float(lengths.max()),
    }
    assert stats["chi"] == 2
    assert abs(row["edge_length_max"] - row["edge_length_min"]) < 1e-8
    platonic_rows.append(row)

mesh_gallery_html = ART_DIRS["html"] / "platonic-solids-mesh-gallery.html"
fig = make_subplots(
    rows=2, cols=3,
    specs=[[{"type": "scene"}, {"type": "scene"}, {"type": "scene"}], [{"type": "scene"}, {"type": "scene"}, {"type": "scene"}]],
    subplot_titles=[f"{name.title()} {data['symbol']}" for name, data in PLATONIC.items()] + [""],
)
colors = ["#ef4444", "#f59e0b", "#14b8a6", "#6366f1", "#a855f7"]
for idx, ((name, data), color) in enumerate(zip(PLATONIC.items(), colors)):
    row = idx // 3 + 1
    col = idx % 3 + 1
    add_polyhedron(fig, data["vertices"], data["faces"], row, col, name, color)
fig.update_layout(title="Five Platonic solids as inspectable meshes", height=760, margin=dict(l=0, r=0, t=70, b=0))
equal_axis_layout(fig)
fig.write_html(mesh_gallery_html, include_plotlyjs=True, full_html=True)

platonic_counts_csv = ART_DIRS["tables"] / "platonic-symbol-counts.csv"
write_csv(platonic_counts_csv, platonic_rows)

display_artifact(mesh_gallery_html, width=920)
display(Markdown("Mesh count table written to `artifacts/chapter-10/tables/platonic-symbol-counts.csv`."))


In [ ]:
def draw_cube_pipeline() -> Path:
    path = ART_DIRS["figures"] / "drawing-model-net-schlegel-pipeline.png"
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))

    ax = axes[0]
    front = np.array([[0, 0], [1.25, 0], [1.25, 1.25], [0, 1.25], [0, 0]])
    back = front + np.array([0.45, 0.36])
    ax.plot(front[:, 0], front[:, 1], color="#111827", lw=2)
    ax.plot(back[:, 0], back[:, 1], color="#111827", lw=2)
    for i in range(4):
        ax.plot([front[i, 0], back[i, 0]], [front[i, 1], back[i, 1]], color="#111827", lw=2)
    ax.set_title("perspective model")
    ax.set_aspect("equal")
    ax.axis("off")

    ax = axes[1]
    squares = [(0, 1), (1, 1), (2, 1), (3, 1), (1, 0), (1, 2)]
    for x, y in squares:
        ax.add_patch(plt.Rectangle((x, y), 1, 1, fill=False, ec="#0f766e", lw=2))
    ax.set_xlim(-0.2, 4.2)
    ax.set_ylim(-0.2, 3.2)
    ax.set_title("net: faces before folding")
    ax.set_aspect("equal")
    ax.axis("off")

    ax = axes[2]
    outer = np.array([[0, 0], [2.4, 0], [2.4, 2.4], [0, 2.4], [0, 0]])
    inner = np.array([[0.75, 0.75], [1.65, 0.75], [1.65, 1.65], [0.75, 1.65], [0.75, 0.75]])
    ax.plot(outer[:, 0], outer[:, 1], color="#1d4ed8", lw=2)
    ax.plot(inner[:, 0], inner[:, 1], color="#1d4ed8", lw=2)
    for a, b in zip(outer[:-1], inner[:-1]):
        ax.plot([a[0], b[0]], [a[1], b[1]], color="#1d4ed8", lw=2)
    ax.text(1.2, -0.25, "outside region is a face", ha="center", va="top", fontsize=9)
    ax.set_title("Schlegel map: incidence")
    ax.set_aspect("equal")
    ax.axis("off")

    fig.suptitle("One cube, three records of the same combinatorics", y=1.02, fontsize=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return path


def draw_schlegel_gallery() -> Path:
    path = ART_DIRS["figures"] / "schlegel-map-gallery.png"
    fig, axes = plt.subplots(2, 3, figsize=(12.5, 8.0))
    axes = axes.ravel()
    for ax, (name, data) in zip(axes, PLATONIC.items()):
        G = nx.Graph()
        G.add_edges_from(edge_pairs(data["faces"]))
        planar, embedding = nx.check_planarity(G)
        assert planar
        try:
            pos = nx.planar_layout(embedding)
        except Exception:
            pos = nx.spring_layout(G, seed=CHAPTER_NO)
        nx.draw_networkx_edges(G, pos, ax=ax, width=1.6, edge_color="#334155")
        nx.draw_networkx_nodes(G, pos, ax=ax, node_size=80, node_color="#f8fafc", edgecolors="#0f172a", linewidths=1.2)
        ax.set_title(f"{name.title()} {data['symbol']}\nV-E+F={len(data['vertices'])}-{len(edge_pairs(data['faces']))}+{len(data['faces'])}=2")
        ax.set_aspect("equal")
        ax.axis("off")
    axes[-1].axis("off")
    fig.suptitle("Schlegel-style planar maps preserve incidence, not metric scale", fontsize=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return path


cube_pipeline_png = draw_cube_pipeline()
schlegel_gallery_png = draw_schlegel_gallery()

schlegel_checks = {
    "chapter": CHAPTER_NO,
    "planar_graphs": [
        {"solid": name, "planar": bool(nx.check_planarity(nx.Graph(edge_pairs(data["faces"])))[0]), "euler": mesh_stats(data["vertices"], data["faces"])["chi"]}
        for name, data in PLATONIC.items()
    ],
}
schlegel_checks_json = ART_DIRS["checks"] / "schlegel-map-checks.json"
write_json(schlegel_checks_json, schlegel_checks)

display_artifact(cube_pipeline_png, width=860)
display_artifact(schlegel_gallery_png, width=860)


## Euler's Formula And The Five Symbols

Euler's formula is the bridge from drawing to classification. A Schlegel diagram turns a simply connected polyhedron into a finite planar map, with the exterior counted as one region. Build such a map edge by edge. If a new edge reaches a new vertex, `V` and `E` both increase by one and `F` is unchanged. If a new edge joins two old vertices and splits a region, `E` and `F` both increase by one and `V` is unchanged. Starting from one vertex and the exterior region, `V - E + F` remains equal to two.

For a regular `{p,q}` polyhedron there are two more counts. Counting edges at vertices gives `qV = 2E`; counting sides of faces gives `pF = 2E`. Together with Euler's formula these force the familiar formulas for `V`, `E`, and `F`. Positivity then forces `(p-2)(q-2) < 4`, leaving only five possible pairs.


In [ ]:
def draw_euler_growth() -> Path:
    path = ART_DIRS["figures"] / "euler-map-growth-invariant.png"
    stages = [
        ("start", [(0, (0.0, 0.0))], [], 1, 0, 1),
        ("new edge to new vertex", [(0, (0.0, 0.0)), (1, (1.0, 0.0))], [(0, 1)], 2, 1, 1),
        ("another new vertex", [(0, (0.0, 0.0)), (1, (1.0, 0.0)), (2, (0.5, 0.85))], [(0, 1), (1, 2)], 3, 2, 1),
        ("old-to-old edge splits a face", [(0, (0.0, 0.0)), (1, (1.0, 0.0)), (2, (0.5, 0.85))], [(0, 1), (1, 2), (2, 0)], 3, 3, 2),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
    for ax, (title, nodes, edges, V, E, F) in zip(axes, stages):
        pos = dict(nodes)
        G = nx.Graph()
        G.add_nodes_from(pos)
        G.add_edges_from(edges)
        nx.draw_networkx_edges(G, pos, ax=ax, width=2.4, edge_color="#1e40af")
        nx.draw_networkx_nodes(G, pos, ax=ax, node_size=220, node_color="#eff6ff", edgecolors="#1e3a8a", linewidths=1.4)
        ax.set_title(title, fontsize=10)
        ax.text(0.5, -0.28, f"V-E+F = {V}-{E}+{F} = {V-E+F}", ha="center", transform=ax.transAxes, fontsize=10)
        ax.set_xlim(-0.25, 1.25)
        ax.set_ylim(-0.25, 1.05)
        ax.set_aspect("equal")
        ax.axis("off")
    fig.suptitle("Euler invariant under the two edge-addition moves", fontsize=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return path


def draw_classification_grid(rows: list[dict]) -> Path:
    path = ART_DIRS["figures"] / "platonic-symbol-classification-grid.png"
    ps = list(range(3, 8))
    qs = list(range(3, 8))
    fig, ax = plt.subplots(figsize=(8, 6.8))
    for ix, p_value in enumerate(ps):
        for iy, q_value in enumerate(qs):
            product_value = (p_value - 2) * (q_value - 2)
            if product_value < 4:
                face = "#dcfce7"
                label = "valid"
            elif product_value == 4:
                face = "#fef3c7"
                label = "flat"
            else:
                face = "#fee2e2"
                label = "overfull"
            ax.add_patch(plt.Rectangle((ix, iy), 1, 1, facecolor=face, edgecolor="#334155", lw=0.8))
            ax.text(ix + 0.5, iy + 0.58, f"{{{p_value},{q_value}}}", ha="center", va="center", fontsize=10, fontweight="bold")
            ax.text(ix + 0.5, iy + 0.28, label, ha="center", va="center", fontsize=8)
    ax.set_xticks([i + 0.5 for i in range(len(ps))], ps)
    ax.set_yticks([i + 0.5 for i in range(len(qs))], qs)
    ax.set_xlabel("p: sides per face")
    ax.set_ylabel("q: faces at a vertex")
    ax.set_xlim(0, len(ps))
    ax.set_ylim(0, len(qs))
    ax.set_title("The solid-angle inequality leaves five regular convex symbols", fontweight="bold")
    ax.set_aspect("equal")
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return path


p_sym, q_sym = sp.symbols("p q", positive=True)
den_sym = 2 * p_sym + 2 * q_sym - p_sym * q_sym
V_sym = 4 * p_sym / den_sym
E_sym = 2 * p_sym * q_sym / den_sym
F_sym = 4 * q_sym / den_sym
symbolic_checks = {
    "V_expr": str(V_sym),
    "E_expr": str(E_sym),
    "F_expr": str(F_sym),
    "euler_simplifies_to": str(sp.simplify(V_sym - E_sym + F_sym)),
    "qV_minus_2E": str(sp.simplify(q_sym * V_sym - 2 * E_sym)),
    "pF_minus_2E": str(sp.simplify(p_sym * F_sym - 2 * E_sym)),
    "positive_denominator_equivalent": "(p - 2)(q - 2) < 4",
}
assert sp.simplify(V_sym - E_sym + F_sym) == 2
assert sp.simplify(q_sym * V_sym - 2 * E_sym) == 0
assert sp.simplify(p_sym * F_sym - 2 * E_sym) == 0

classification_rows = []
for p_value in range(3, 9):
    for q_value in range(3, 9):
        den = 2 * p_value + 2 * q_value - p_value * q_value
        status = "valid" if den > 0 else "flat" if den == 0 else "overfull"
        classification_rows.append({"p": p_value, "q": q_value, "denominator": den, "status": status})

valid_symbols = [(row["p"], row["q"]) for row in classification_rows if row["status"] == "valid"]
assert valid_symbols == [(3, 3), (3, 4), (3, 5), (4, 3), (5, 3)]

for row in platonic_rows:
    p_value, q_value = row["p"], row["q"]
    den = 2 * p_value + 2 * q_value - p_value * q_value
    assert row["V"] == int(4 * p_value / den)
    assert row["E"] == int(2 * p_value * q_value / den)
    assert row["F"] == int(4 * q_value / den)
    assert q_value * row["V"] == 2 * row["E"] == p_value * row["F"]

classification_grid_png = draw_classification_grid(classification_rows)
euler_growth_png = draw_euler_growth()
classification_csv = ART_DIRS["tables"] / "platonic-classification-grid.csv"
euler_checks_json = ART_DIRS["checks"] / "euler-classification-checks.json"
write_csv(classification_csv, classification_rows)
write_json(euler_checks_json, {
    "chapter": CHAPTER_NO,
    "symbolic_checks": symbolic_checks,
    "valid_symbols": valid_symbols,
    "platonic_counts": platonic_rows,
})

display_artifact(euler_growth_png, width=860)
display_artifact(classification_grid_png, width=720)


## Radii, Central Angles, And Dihedral Angles

The source introduces a characteristic tetrahedron inside a regular `{p,q}` polyhedron. Its vertices are a polyhedron vertex, an edge midpoint, a face center, and the common center of the solid. This tetrahedron is useful because its relevant faces are right triangles. If the edge length is one, write `l = 1/2` for half an edge. The formulas below use the chapter's three radii:

- `R0`: circumradius, from center to a vertex;
- `R1`: midradius, from center to an edge midpoint;
- `R2`: inradius, from center to a face plane.

The angular quantities are also split by the characteristic tetrahedron. The central half-angle `phi` is controlled by `cos(phi)=cos(pi/p)/sin(pi/q)`. The complementary piece `psi` gives the dihedral angle `pi - 2 psi`, equivalently `2 asin(cos(pi/q)/sin(pi/p))`. The check is not a visual impression: it verifies the formulas for all five symbols, including the dual identity that swaps `p` and `q`.


In [ ]:
def radii_angles_for_symbol(p_value: int, q_value: int, half_edge: float = 0.5) -> dict[str, float | int]:
    k_sq = math.sin(math.pi / q_value) ** 2 - math.cos(math.pi / p_value) ** 2
    k = math.sqrt(k_sq)
    phi_angle = math.acos(math.cos(math.pi / p_value) / math.sin(math.pi / q_value))
    psi_angle = math.acos(math.cos(math.pi / q_value) / math.sin(math.pi / p_value))
    return {
        "p": p_value,
        "q": q_value,
        "k": k,
        "R0_circumradius": half_edge * math.sin(math.pi / q_value) / k,
        "R1_midradius": half_edge * math.cos(math.pi / p_value) / k,
        "R2_inradius": half_edge * (math.cos(math.pi / q_value) / math.tan(math.pi / p_value)) / k,
        "phi_degrees": math.degrees(phi_angle),
        "psi_degrees": math.degrees(psi_angle),
        "dihedral_degrees": math.degrees(math.pi - 2 * psi_angle),
        "angular_defect_degrees": math.degrees(2 * math.pi - q_value * (1 - 2 / p_value) * math.pi),
    }


radii_rows = []
for name, data in PLATONIC.items():
    row = {"solid": name, "symbol": data["symbol"], **radii_angles_for_symbol(data["p"], data["q"])}
    expected_defect = 720.0 / len(data["vertices"])
    assert abs(row["angular_defect_degrees"] - expected_defect) < 1e-9
    radii_rows.append(row)

radii_by_symbol = {(row["p"], row["q"]): row for row in radii_rows}
for p_value, q_value in [(3, 4), (4, 3), (3, 5), (5, 3), (3, 3)]:
    assert abs(radii_by_symbol[(p_value, q_value)]["phi_degrees"] - radii_by_symbol[(q_value, p_value)]["psi_degrees"]) < 1e-9

assert abs(radii_by_symbol[(3, 3)]["dihedral_degrees"] + radii_by_symbol[(3, 4)]["dihedral_degrees"] - 180.0) < 1e-9

radii_csv = ART_DIRS["tables"] / "radii-angle-table.csv"
radii_checks_json = ART_DIRS["checks"] / "radii-angle-checks.json"
write_csv(radii_csv, radii_rows)
write_json(radii_checks_json, {
    "chapter": CHAPTER_NO,
    "edge_length": 1,
    "rows": radii_rows,
    "duality_phi_psi_checked": True,
    "tetrahedron_octahedron_dihedral_sum_degrees": radii_by_symbol[(3, 3)]["dihedral_degrees"] + radii_by_symbol[(3, 4)]["dihedral_degrees"],
})

radii_png = ART_DIRS["figures"] / "radii-and-dihedral-comparison.png"
labels = [row["symbol"] for row in radii_rows]
x = np.arange(len(labels))
width = 0.24
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))
ax1.bar(x - width, [row["R0_circumradius"] for row in radii_rows], width, label="R0 circumradius", color="#2563eb")
ax1.bar(x, [row["R1_midradius"] for row in radii_rows], width, label="R1 midradius", color="#16a34a")
ax1.bar(x + width, [row["R2_inradius"] for row in radii_rows], width, label="R2 inradius", color="#dc2626")
ax1.set_xticks(x, labels)
ax1.set_ylabel("radius for edge length 1")
ax1.set_title("Three concentric radii")
ax1.legend(fontsize=8)
ax1.grid(axis="y", alpha=0.2)

ax2.bar(x, [row["dihedral_degrees"] for row in radii_rows], color="#7c3aed", label="dihedral")
ax2.plot(x, [row["angular_defect_degrees"] for row in radii_rows], color="#ea580c", marker="o", label="vertex angle defect")
ax2.set_xticks(x, labels)
ax2.set_ylabel("degrees")
ax2.set_title("Angles: face hinge and vertex defect")
ax2.legend(fontsize=8)
ax2.grid(axis="y", alpha=0.2)
fig.suptitle("Radii and angles from the characteristic tetrahedron", fontweight="bold")
fig.tight_layout()
fig.savefig(radii_png, dpi=180, bbox_inches="tight")
plt.close(fig)

display_artifact(radii_png, width=860)
display(Markdown("The numeric table is saved as `artifacts/chapter-10/tables/radii-angle-table.csv`."))


## Reciprocal Polyhedra

The reciprocal of `{p,q}` is `{q,p}`. Computationally, a convenient model is made from face centers: place a point at each face center of the original solid, and connect two such points when the corresponding original faces share an edge. This swaps vertices and faces while keeping the same number of edges.

The reciprocal construction is not merely a count. For a regular solid, the edge of the reciprocal joining two adjacent face centers is perpendicular to the original edge where those faces meet. The overlay below tests that perpendicularity for the cube-octahedron pair, the self-reciprocal tetrahedron, and the dodecahedron-icosahedron pair.


In [ ]:
def dual_edge_data(vertices: np.ndarray, faces: list[list[int]]) -> tuple[np.ndarray, list[tuple[int, int]], dict[tuple[int, int], list[int]]]:
    centers = face_centers(vertices, faces)
    incidence: dict[tuple[int, int], list[int]] = defaultdict(list)
    for face_index, face in enumerate(faces):
        for a, b in zip(face, face[1:] + face[:1]):
            incidence[tuple(sorted((a, b)))].append(face_index)
    dual_edges = [tuple(face_ids) for face_ids in incidence.values() if len(face_ids) == 2]
    return centers, sorted(dual_edges), incidence


def dual_orthogonality_residual(vertices: np.ndarray, faces: list[list[int]]) -> float:
    centers, _, incidence = dual_edge_data(vertices, faces)
    residuals = []
    for (a, b), face_ids in incidence.items():
        if len(face_ids) != 2:
            continue
        original_direction = vertices[b] - vertices[a]
        dual_direction = centers[face_ids[1]] - centers[face_ids[0]]
        denom = np.linalg.norm(original_direction) * np.linalg.norm(dual_direction)
        residuals.append(abs(float(np.dot(original_direction, dual_direction))) / denom)
    return max(residuals) if residuals else 0.0


def dual_edge_trace(centers: np.ndarray, dual_edges: list[tuple[int, int]], name: str, color: str) -> go.Scatter3d:
    xs: list[float | None] = []
    ys: list[float | None] = []
    zs: list[float | None] = []
    for a, b in dual_edges:
        xs.extend([float(centers[a, 0]), float(centers[b, 0]), None])
        ys.extend([float(centers[a, 1]), float(centers[b, 1]), None])
        zs.extend([float(centers[a, 2]), float(centers[b, 2]), None])
    return go.Scatter3d(x=xs, y=ys, z=zs, mode="lines", line=dict(color=color, width=6), name=name, showlegend=False)


reciprocal_pairs = [("cube", "octahedron"), ("tetrahedron", "tetrahedron"), ("dodecahedron", "icosahedron")]
reciprocal_html = ART_DIRS["html"] / "reciprocal-polyhedra-face-center-overlays.html"
fig = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "scene"}, {"type": "scene"}, {"type": "scene"}]],
    subplot_titles=["cube -> octahedron", "tetrahedron -> tetrahedron", "dodecahedron -> icosahedron"],
)
reciprocal_rows = []
for col, (original_name, reciprocal_name) in enumerate(reciprocal_pairs, start=1):
    data = PLATONIC[original_name]
    vertices, faces = data["vertices"], data["faces"]
    centers, dual_edges, _ = dual_edge_data(vertices, faces)
    residual = dual_orthogonality_residual(vertices, faces)
    fig.add_trace(plotly_mesh_trace(vertices, faces, original_name, "#94a3b8", opacity=0.22), row=1, col=col)
    fig.add_trace(plotly_edges_trace(vertices, faces, original_name + " edges", color="#334155", width=3), row=1, col=col)
    fig.add_trace(dual_edge_trace(centers, dual_edges, reciprocal_name + " dual edges", color="#dc2626"), row=1, col=col)
    fig.add_trace(go.Scatter3d(x=centers[:, 0], y=centers[:, 1], z=centers[:, 2], mode="markers", marker=dict(size=4, color="#dc2626"), showlegend=False), row=1, col=col)
    original_stats = mesh_stats(vertices, faces)
    reciprocal_rows.append({
        "original": original_name,
        "original_symbol": data["symbol"],
        "reciprocal": reciprocal_name,
        "reciprocal_symbol": PLATONIC[reciprocal_name]["symbol"],
        "original_V": original_stats["V"],
        "original_E": original_stats["E"],
        "original_F": original_stats["F"],
        "dual_vertices_from_faces": int(len(centers)),
        "dual_edges_from_shared_original_edges": int(len(dual_edges)),
        "max_edge_dual_orthogonality_residual": residual,
    })
    assert residual < 1e-10
    assert len(centers) == original_stats["F"]
    assert len(dual_edges) == original_stats["E"]
fig.update_layout(title="Reciprocal polyhedra from face centers", height=530, margin=dict(l=0, r=0, t=65, b=0))
equal_axis_layout(fig)
fig.write_html(reciprocal_html, include_plotlyjs=True, full_html=True)

reciprocal_checks_json = ART_DIRS["checks"] / "reciprocal-polyhedra-checks.json"
write_json(reciprocal_checks_json, {"chapter": CHAPTER_NO, "pairs": reciprocal_rows})

display_artifact(reciprocal_html, width=920)
display(Markdown("Every reported dual edge is perpendicular to its corresponding original edge to numerical precision."))


## Applied Lab: Euler Survives When Regularity Fails

Euler's formula is powerful but deliberately weak: it knows the topology of the map, not the metric symmetry of a Platonic solid. A rectangular box has exactly the same vertex-edge-face counts as a cube, and its face-center reciprocal has the graph of an octahedron. Still, it is not regular: the edge lengths split into three sizes and the face-center distances from the center are unequal.

This lab is a useful guardrail for the chapter. If a computation proves only `V - E + F = 2`, it has not proved regularity. To prove regularity we must also check equal faces, equal edges, and equal vertex neighborhoods, or use the `{p,q}` formulas under their regularity hypotheses.


In [ ]:
def cuboid_mesh(a: float, b: float, c: float) -> tuple[np.ndarray, list[list[int]]]:
    vertices = np.array([
        [-a / 2, -b / 2, -c / 2], [a / 2, -b / 2, -c / 2], [a / 2, b / 2, -c / 2], [-a / 2, b / 2, -c / 2],
        [-a / 2, -b / 2, c / 2], [a / 2, -b / 2, c / 2], [a / 2, b / 2, c / 2], [-a / 2, b / 2, c / 2],
    ], dtype=float)
    faces = [[0, 1, 2, 3], [4, 7, 6, 5], [0, 4, 5, 1], [1, 5, 6, 2], [2, 6, 7, 3], [3, 7, 4, 0]]
    return vertices, faces


box_vertices, box_faces = cuboid_mesh(1.0, 1.45, 2.1)
box_stats = mesh_stats(box_vertices, box_faces)
box_lengths = np.round(edge_lengths(box_vertices, box_faces), 10)
box_centers = face_centers(box_vertices, box_faces)
box_center_radii = np.linalg.norm(box_centers, axis=1)
box_lab = {
    "chapter": CHAPTER_NO,
    "shape": "rectangular box",
    "V": box_stats["V"],
    "E": box_stats["E"],
    "F": box_stats["F"],
    "euler_characteristic": box_stats["chi"],
    "distinct_edge_lengths": sorted(float(x) for x in np.unique(box_lengths)),
    "face_center_radius_min": float(box_center_radii.min()),
    "face_center_radius_max": float(box_center_radii.max()),
    "regularity_failure": "Euler characteristic and reciprocal graph survive, but metric equality of edge lengths and concentric face-center radius fails.",
}
assert box_lab["euler_characteristic"] == 2
assert len(box_lab["distinct_edge_lengths"]) == 3
assert box_lab["face_center_radius_max"] - box_lab["face_center_radius_min"] > 0.4

lab_json = ART_DIRS["checks"] / "regularity-vs-euler-lab.json"
write_json(lab_json, box_lab)

lab_png = ART_DIRS["figures"] / "regularity-vs-euler-lab.png"
fig = plt.figure(figsize=(9, 4.2))
for idx, (title, dims) in enumerate([("cube", (1, 1, 1)), ("rectangular box", (1.0, 1.45, 2.1))], start=1):
    ax = fig.add_subplot(1, 2, idx, projection="3d")
    vertices, faces = cuboid_mesh(*dims)
    for a, b in edge_pairs(faces):
        ax.plot(*zip(vertices[a], vertices[b]), color="#0f172a", lw=1.8)
    ax.scatter(vertices[:, 0], vertices[:, 1], vertices[:, 2], color="#dc2626", s=18)
    ax.set_title(f"{title}\nV-E+F=2")
    ax.set_box_aspect((1, 1, 1))
    ax.set_axis_off()
fig.suptitle("Same Euler count, different metric regularity", fontweight="bold")
fig.tight_layout()
fig.savefig(lab_png, dpi=180, bbox_inches="tight")
plt.close(fig)

display_artifact(lab_png, width=760)
box_lab


## Takeaways

- Pyramids, prisms, and antiprisms explain where four of the five regular solids appear naturally: tetrahedron as a triangular pyramid, cube as a square prism, octahedron as a triangular antiprism, and icosahedron by combining a pentagonal antiprism with two pentagonal pyramids.
- A Schlegel diagram is a planar map of incidence, not a metric duplicate of the solid. Counting the exterior region makes Euler's invariant visible.
- The relations `qV = 2E = pF` plus `V - E + F = 2` force the formulas for `V`, `E`, and `F`; the inequality `(p-2)(q-2) < 4` leaves only `{3,3}`, `{4,3}`, `{3,4}`, `{5,3}`, and `{3,5}`.
- The characteristic tetrahedron packages the metric data: circumradius, midradius, inradius, central half-angle, and dihedral angle all come from right-triangle relations.
- Reciprocation swaps `p` and `q`: cube with octahedron, dodecahedron with icosahedron, and tetrahedron with itself. Face centers provide a direct computational model of that swap.
- Euler's formula is necessary for these solids but not sufficient for regularity; the final lab shows the metric checks that Euler cannot see.


In [ ]:
visual_artifacts = [
    construction_html,
    mesh_gallery_html,
    cube_pipeline_png,
    schlegel_gallery_png,
    euler_growth_png,
    classification_grid_png,
    radii_png,
    reciprocal_html,
    lab_png,
]
check_artifacts = [
    ART_DIRS["checks"] / "visual-storyboard.json",
    construction_checks_json,
    schlegel_checks_json,
    euler_checks_json,
    radii_checks_json,
    reciprocal_checks_json,
    lab_json,
]
table_artifacts = [
    construction_counts_csv,
    platonic_counts_csv,
    classification_csv,
    radii_csv,
]

for stale_path in STALE_ARTIFACTS:
    assert not stale_path.exists(), f"stale scaffold artifact remains: {stale_path}"

for row in platonic_rows:
    assert row["chi"] == 2
    assert row["q"] * row["V"] == 2 * row["E"] == row["p"] * row["F"]
    assert (row["p"] - 2) * (row["q"] - 2) < 4

assert set(tuple(x) for x in valid_symbols) == {(3, 3), (4, 3), (3, 4), (5, 3), (3, 5)}
assert all(pair["max_edge_dual_orthogonality_residual"] < 1e-10 for pair in reciprocal_rows)
assert abs(radii_by_symbol[(3, 3)]["dihedral_degrees"] + radii_by_symbol[(3, 4)]["dihedral_degrees"] - 180.0) < 1e-9

visual_summary = {
    "chapter": CHAPTER_NO,
    "title": "The Five Platonic Solids",
    "source_span": SOURCE_SPAN,
    "visuals": [str(path.relative_to(BOOK_ROOT).as_posix()) for path in visual_artifacts],
    "checks": [str(path.relative_to(BOOK_ROOT).as_posix()) for path in check_artifacts],
    "tables": [str(path.relative_to(BOOK_ROOT).as_posix()) for path in table_artifacts],
    "core_identities_checked": [
        "V - E + F = 2",
        "qV = 2E = pF",
        "(p-2)(q-2) < 4 for the five valid symbols",
        "4*pi/V equals vertex angle defect",
        "phi(p,q) = psi(q,p) under reciprocation",
        "face-center reciprocal edges are perpendicular to original edges",
    ],
}
visual_summary_path = ART_DIRS["checks"] / "visual_summary.json"
write_json(visual_summary_path, visual_summary)

final_sanity_path = ART_DIRS["checks"] / "final-sanity.json"
pre_final_artifacts = visual_artifacts + check_artifacts + table_artifacts + [visual_summary_path]
assert_artifacts(pre_final_artifacts, min_bytes=100)

final_sanity = {
    "chapter": CHAPTER_NO,
    "visual_artifact_count": len(visual_artifacts),
    "check_artifact_count": len(check_artifacts) + 2,
    "table_artifact_count": len(table_artifacts),
    "stale_scaffold_artifacts_absent": all(not p.exists() for p in STALE_ARTIFACTS),
    "platonic_symbols": [row["symbol"] for row in platonic_rows],
    "max_reciprocal_orthogonality_residual": max(pair["max_edge_dual_orthogonality_residual"] for pair in reciprocal_rows),
    "smallest_artifact_bytes_before_final": min(path.stat().st_size for path in pre_final_artifacts),
    "source_span": SOURCE_SPAN,
}
write_json(final_sanity_path, final_sanity)
assert_artifacts(pre_final_artifacts + [final_sanity_path], min_bytes=100)
final_sanity
